# 🤖 Agentic RAG — Let the LLM Decide What to Search

## The Difference

```
Basic RAG:    User asks → system searches once → LLM answers
              (Fixed pipeline, one search)

Agentic RAG:  User asks → LLM thinks → LLM decides to search
              → LLM reads results → LLM decides to search again?
              → LLM has enough → LLM answers
              (LLM is in control, can search multiple times)
```

## How It Works — The Tool Use Pattern

The LLM is given a **search tool** it can call whenever it needs.

```
User: "What's Alice's opinion on the database?"

LLM: I need to find who Alice is first.
     → CALLS search("Alice role")
     → Gets: "Alice is the project manager"

LLM: Now I need her database opinion.
     → CALLS search("Alice database")
     → Gets: "Alice approved PostgreSQL in Q3"

LLM: I have enough. Answer: Alice approved using PostgreSQL.
```

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Knowledge base
docs = [
    "Alice is the project manager for Project Phoenix.",
    "Project Phoenix is a customer data platform launched in 2023.",
    "Bob is the lead engineer on Project Phoenix.",
    "Bob wrote a design doc recommending PostgreSQL for the database.",
    "Alice approved Bob's PostgreSQL recommendation in the Q3 review.",
    "The Q3 review was held in September 2023.",
]

doc_embeddings = model.encode(docs)
print(f"Knowledge base: {len(docs)} docs")

In [ ]:
# The search tool — this is what the LLM can call

def search_tool(query, top_k=2):
    """Search the knowledge base. Returns top_k relevant documents."""
    q_emb = model.encode(query)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = [docs[i] for i in top_idx]
    return results

# Test the tool
print("Tool test — search('Alice role'):")
for r in search_tool("Alice role"):
    print(f"  → {r}")

In [ ]:
# Simulated Agentic RAG loop
# In production: the LLM generates the search queries
# Here: we hardcode the agent's reasoning to show the pattern clearly

question = "What did the project manager say about the database?"

print(f"Question: {question}")
print("=" * 55)
print()

# --- Agent Step 1 ---
print("Agent: I need to find who the project manager is.")
search_q1 = "who is the project manager"
results1 = search_tool(search_q1)
print(f"  Searched: '{search_q1}'")
for r in results1:
    print(f"  Got: {r}")
print()

# Agent reads results and decides to search again
print("Agent: Alice is the project manager. Now I'll search for her database opinion.")
search_q2 = "Alice database decision"
results2 = search_tool(search_q2)
print(f"  Searched: '{search_q2}'")
for r in results2:
    print(f"  Got: {r}")
print()

# Agent has enough information
print("Agent: I have enough. Generating final answer.")
print()
all_context = results1 + results2
print("Final Answer:")
print("  Alice (the project manager) approved using PostgreSQL for the database.")

In [ ]:
# How to implement this with a real LLM (Anthropic tool use)

example_code = '''
from anthropic import Anthropic
client = Anthropic()

# Define the search tool for the LLM
tools = [{
    "name": "search",
    "description": "Search the knowledge base for relevant documents",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The search query"}
        },
        "required": ["query"]
    }
}]

messages = [{"role": "user", "content": question}]

# Agent loop — keep going until LLM stops calling tools
while True:
    response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=500, tools=tools,
        messages=messages
    )

    if response.stop_reason == "end_turn":  # LLM is done
        print(response.content[0].text)
        break

    # LLM called a tool — execute it and feed results back
    for block in response.content:
        if block.type == "tool_use" and block.name == "search":
            results = search_tool(block.input["query"])
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(results)
            }]})
'''

print("Real implementation with Anthropic tool use:")
print(example_code)

## Key Takeaways 🎯

- **Basic RAG**: one fixed search per question
- **Agentic RAG**: LLM searches as many times as needed
- The LLM uses a **search tool** — just a Python function it can call
- Best for: complex questions, multi-step reasoning, unknowns
- Trade-off: slower (multiple LLM calls) but much more accurate

---
Next: `02_Self_RAG.ipynb` — what if the LLM checks whether its own retrieval was good enough?